# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adithyajupally/flyrank-ml-internship-jupally/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will give each page a simple score based on its search position and CTR. I want to find pages that rank reasonably well but get fewer clicks than expected. Pages with a higher opportunity score will be placed higher in the review list.

The rule will use these reason codes:

- `GOOD_POSITION_LOW_CTR` — the page has a good search position but a relatively low CTR.
- `HIGH_IMPRESSIONS_LOW_CTR` — the page gets many impressions but a relatively low CTR.
- `LOW_CTR_FOR_POSITION` — the page's CTR is low compared with pages in a similar position.

In [2]:
!git clone https://github.com/adithyajupally/flyrank-ml-internship-jupally.git
%cd flyrank-ml-internship-jupally

Cloning into 'flyrank-ml-internship-jupally'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 47), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.87 MiB | 6.35 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-internship-jupally


In [3]:
# main columns we'll use for the baseline rule

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

reason_codes = ["GOOD_POSITION_LOW_CTR","HIGH_IMPRESSIONS_LOW_CTR","LOW_CTR_FOR_POSITION"]

print("Baseline rule reason codes:")
for code in reason_codes:
    print("->", code)

print("\nRows in dataset:", len(df))

Baseline rule reason codes:
-> GOOD_POSITION_LOW_CTR
-> HIGH_IMPRESSIONS_LOW_CTR
-> LOW_CTR_FOR_POSITION

Rows in dataset: 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I created a simple opportunity score using three signals:

- Better search position
- Lower CTR
- Higher impressions

Pages that rank reasonably well, receive many impressions, and have a low CTR are given a higher score because they may have room for improvement.

In [4]:
baseline = df.copy()

# Normalize signals to 0-1 range
baseline["position_score"] = (baseline["avg_position"].max() - baseline["avg_position"]) / (baseline["avg_position"].max() - baseline["avg_position"].min())

baseline["impression_score"] = (baseline["impressions_90d"] - baseline["impressions_90d"].min()) / (baseline["impressions_90d"].max() - baseline["impressions_90d"].min())

baseline["low_ctr_score"] = 1 - baseline["ctr"]

# Final opportunity score
baseline["opportunity_score"] = (0.4 * baseline["position_score"] + 0.4 * baseline["low_ctr_score"] + 0.2 * baseline["impression_score"])

# Rank pages
baseline = baseline.sort_values("opportunity_score",ascending=False)

# Reason code
baseline["reason_code"] = "LOW_CTR_FOR_POSITION"

baseline.loc[(baseline["avg_position"] <= 20) & (baseline["ctr"] < 0.20),"reason_code"] = "GOOD_POSITION_LOW_CTR"

baseline.loc[(baseline["impressions_90d"] > baseline["impressions_90d"].median()) &(baseline["ctr"] < 0.20),"reason_code"] = "HIGH_IMPRESSIONS_LOW_CTR"

import os
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
output_file = "work/outputs/baseline_action_score.csv"
baseline.to_csv(output_file, index=False)

print("Saved:", output_file)

baseline[["content_id","avg_position","ctr","impressions_90d","opportunity_score","reason_code"]].head(10)

Saved: work/outputs/baseline_action_score.csv


,content_id,avg_position,ctr,impressions_90d,opportunity_score,reason_code
6653,content_5fe46e04994d,4.2,0.14,517715,0.937143,HIGH_IMPRESSIONS_LOW_CTR
26844,content_8c19996aa890,2.5,0.15,509252,0.932649,HIGH_IMPRESSIONS_LOW_CTR
19636,content_2cb567c3c89b,22.2,0.10,497727,0.916033,HIGH_IMPRESSIONS_LOW_CTR
17812,content_aaef01a50def,5.4,0.25,517109,0.890950,LOW_CTR_FOR_POSITION
7678,content_8451fc6f034d,2.3,0.03,272144,0.889377,HIGH_IMPRESSIONS_LOW_CTR
3394,content_36ff89c8214e,7.3,0.05,295097,0.882081,HIGH_IMPRESSIONS_LOW_CTR
7445,content_c8e9d6ab9013,9.7,0.00,208678,0.864778,HIGH_IMPRESSIONS_LOW_CTR
29879,content_1a9e894be2e2,4.0,0.23,416180,0.862245,LOW_CTR_FOR_POSITION
6903,content_c84a0ab98e90,7.8,0.03,223271,0.861518,HIGH_IMPRESSIONS_LOW_CTR
26531,content_cb112fce36be,5.6,0.16,309910,0.846579,HIGH_IMPRESSIONS_LOW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 pages from the baseline score.

These pages are suggested for further review because they have a higher opportunity score.

The reason code shows why each page was selected.

This is only a simple baseline check, so the results may not always be correct.

In [5]:
# Take the top 20 pages from the ranked baseline
top_20 = baseline.head(20).copy()

# Add simple review information
top_20["action"] = "REVIEW"
top_20["confidence_note"] = "Directional baseline signal"
top_20["what_could_make_it_wrong"] = ("Low CTR may be normal for this page or other factors may explain the result.")

# Show the top 20 review queue
top_20[["content_id","avg_position","ctr","impressions_90d","opportunity_score","reason_code","action","confidence_note","what_could_make_it_wrong"]]

,content_id,avg_position,ctr,impressions_90d,opportunity_score,reason_code,action,confidence_note,what_could_make_it_wrong
6653,content_5fe46e04994d,4.2,0.14,517715,0.937143,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
26844,content_8c19996aa890,2.5,0.15,509252,0.932649,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
19636,content_2cb567c3c89b,22.2,0.10,497727,0.916033,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
17812,content_aaef01a50def,5.4,0.25,517109,0.890950,LOW_CTR_FOR_POSITION,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
7678,content_8451fc6f034d,2.3,0.03,272144,0.889377,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
3394,content_36ff89c8214e,7.3,0.05,295097,0.882081,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
7445,content_c8e9d6ab9013,9.7,0.00,208678,0.864778,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
29879,content_1a9e894be2e2,4.0,0.23,416180,0.862245,LOW_CTR_FOR_POSITION,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
6903,content_c84a0ab98e90,7.8,0.03,223271,0.861518,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...
26531,content_cb112fce36be,5.6,0.16,309910,0.846579,HIGH_IMPRESSIONS_LOW_CTR,REVIEW,Directional baseline signal,Low CTR may be normal for this page or other f...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

I checked the top picks to see if any of them look unusual or wrong.

Some pages may get a high score even though their low CTR is normal. So these results should only be used for review, not as a final decision.

I also checked that the score only uses the available page data and does not use future information or product flags.

In [6]:
# Check the top 20 pages
print("Top 20 pages checked:", len(top_20))

# Check which columns were used in the score
score_columns = ["avg_position","ctr","impressions_90d"]

print("\nColumns used for scoring:")
for col in score_columns:
    print("->", col)

# Check for product or future-looking columns
product_columns = [col for col in df.columns if "product" in col.lower()]
future_columns = [col for col in df.columns if "future" in col.lower() or "next" in col.lower()]

print("\nProduct-related columns found:", product_columns)
print("Future-looking columns found:", future_columns)

Top 20 pages checked: 20

Columns used for scoring:
-> avg_position
-> ctr
-> impressions_90d

Product-related columns found: []
Future-looking columns found: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.